# LLM smoke test (Kaggle / Colab)

Checks four things before we build on it: a GPU is attached, Ollama installs, **IBM Granite 4** downloads, and it can return **schema-constrained JSON** (the agent depends on that).

**Kaggle:** right panel -> Settings -> Accelerator = *GPU T4 x2* (or T4), *Internet = On* (needs phone verification). **Colab:** Runtime -> Change runtime type -> T4 GPU.

Run the cells top to bottom and paste the outputs of cells 3 and 4 back to me. If any cell errors, paste the error.

In [ ]:
# 1) Install Ollama (Internet must be On)
!apt-get install -y -q zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2) Start the server in the background and download the model (~2 GB)
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
!ollama pull granite4:micro

In [ ]:
# 3) Free-text call + speed
!pip install -q ollama
import ollama, time
t = time.time()
r = ollama.chat(model="granite4:micro", messages=[{"role": "user", "content": "In one sentence: why do telecom customers churn?"}])
print(r["message"]["content"])
print(f"\nlatency: {time.time()-t:.1f}s | tokens/s: {r['eval_count']/(r['eval_duration']/1e9):.1f}")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# 4) Schema-constrained JSON (what the agent will rely on)
import json
schema = {"type": "object", "properties": {"offer": {"type": "string"}, "discount_pct": {"type": "integer"}}, "required": ["offer", "discount_pct"]}
r = ollama.chat(model="granite4:micro", format=schema,
                messages=[{"role": "user", "content": "Suggest a retention offer for a month-to-month customer. Return JSON."}])
print(json.loads(r["message"]["content"]))